# T2.2 – Fine-Tuning YOLO on KITTI (Three Configurations)

## 1. Setup & Imports

In [ ]:
# 1. Install YOLO
!pip install -q -U ultralytics

# Standard & file system libraries
import os
import pandas as pd

# Visualisation & display
from IPython.display import Image, display

# Deep learning / YOLO
from ultralytics import YOLO

from google.colab import drive
drive.mount('/content/drive')

print("Environment setup complete. All libraries imported.")


## 2. Load & Extract the Dataset

In [ ]:
# Replace 'kitti3k_yolo.zip' with the exact name of the zip file if it is different.
# If the zip file is inside a specific folder in Drive, include that folder path
# (e.g. /content/drive/MyDrive/University/kitti3k_yolo.zip)
!unzip -q "/content/drive/MyDrive/kitti3k_yolo.zip" -d "/content/kitti3k_yolo/"

print("Unzipping complete!")


### Flatten the Dataset Folder Structure

In [ ]:
# Move everything from the inner folder up one level
!mv /content/kitti3k_yolo/kitti3k_yolo/* /content/kitti3k_yolo/


## 3. Configuration 1: Partial Freeze (freeze=10, 20 epochs)

In [ ]:
# 1. Define the dataset config path
dataset_yaml = "/content/kitti3k_yolo/kitti3k.yaml"

# 2. Load fresh pre-trained YOLO11n weights
model_cfg1 = YOLO("yolo11n.pt")

print("--- Starting Training: Configuration 1 ---")
# 3. Train using the brief's baseline parameters (partial freeze, 20 epochs)
train_results_cfg1 = model_cfg1.train(
    data=dataset_yaml,
    epochs=20,
    imgsz=640,
    batch=16,
    lr0=0.001,
    freeze=10,
    workers=2,
    cache="ram",
    device=0,
    exist_ok=True,
    pretrained=True,
    name="cfg1_fine_tune"
)


## 4. Configuration 2: Full Fine-Tuning (freeze=0, 20 epochs)

In [ ]:
# --- Configuration 2: full backbone fine-tuning (freeze=0) ---
print("--- Starting Training: Configuration 2 ---")

model_cfg2 = YOLO("yolo11n.pt")

train_results_cfg2 = model_cfg2.train(
    data=dataset_yaml,
    epochs=20,
    imgsz=640,
    batch=16,
    lr0=0.001,
    freeze=0,          # Unfreeze all layers for end-to-end training
    workers=2,
    cache="ram",
    device=0,
    exist_ok=True,
    pretrained=True,
    name="cfg2_fine_tune"
)


## 5. Configuration 3: Extended Training & Higher LR (freeze=0, 30 epochs, lr0=0.005)

In [ ]:
# --- Configuration 3: extended training & higher LR (freeze=0, lr0=0.005, epochs=30) ---
print("--- Starting Training: Configuration 3 ---")

model_cfg3 = YOLO("yolo11n.pt")

train_results_cfg3 = model_cfg3.train(
    data=dataset_yaml,
    epochs=30,          # Increased training duration
    imgsz=640,
    batch=16,
    lr0=0.005,          # Slightly higher initial learning rate
    freeze=0,           # Full network fine-tuning
    workers=2,
    cache="ram",
    device=0,
    exist_ok=True,
    pretrained=True,
    name="cfg3_fine_tune"
)


## 6. Compare Configurations on the Validation Set

In [ ]:
# Load models from their best saved weights
best_cfg1 = YOLO("runs/detect/cfg1_fine_tune/weights/best.pt")
best_cfg2 = YOLO("runs/detect/cfg2_fine_tune/weights/best.pt")
best_cfg3 = YOLO("runs/detect/cfg3_fine_tune/weights/best.pt")

# Evaluate each on the validation set
m1 = best_cfg1.val(data=dataset_yaml, split="val", imgsz=640, conf=0.001, iou=0.7, exist_ok=True)
m2 = best_cfg2.val(data=dataset_yaml, split="val", imgsz=640, conf=0.001, iou=0.7, exist_ok=True)
m3 = best_cfg3.val(data=dataset_yaml, split="val", imgsz=640, conf=0.001, iou=0.7, exist_ok=True)

# Build the summary DataFrame (indices: 0=Cyclist, 1=Pedestrian, 2=Car)
summary_data = {
    "Config": [
        "Config 1 (Freeze=10, Ep=20)",
        "Config 2 (Freeze=0, Ep=20)",
        "Config 3 (Freeze=0, Ep=30, LR=0.005)"
    ],
    "Pedestrian AP50":    [m1.box.ap50[1], m2.box.ap50[1], m3.box.ap50[1]],
    "Pedestrian AP50-95": [m1.box.maps[1], m2.box.maps[1], m3.box.maps[1]],
    "Cyclist AP50":       [m1.box.ap50[0], m2.box.ap50[0], m3.box.ap50[0]],
    "Cyclist AP50-95":    [m1.box.maps[0], m2.box.maps[0], m3.box.maps[0]],
    "Car AP50":           [m1.box.ap50[2], m2.box.ap50[2], m3.box.ap50[2]],
    "Car AP50-95":        [m1.box.maps[2], m2.box.maps[2], m3.box.maps[2]],
    "mAP50 (Val)":        [m1.box.map50,   m2.box.map50,   m3.box.map50],
    "mAP50-95 (Val)":     [m1.box.map,     m2.box.map,     m3.box.map]
}

df_summary = pd.DataFrame(summary_data)
print("\n--- TASK 2.2 VALIDATION PERFORMANCE COMPARISON ---")
display(df_summary)


## 7. Evaluate the Best Configuration on the Test Set

In [ ]:
print("--- Evaluating Best Model (Config 3) on the TEST Set ---")

best_model = YOLO("runs/detect/cfg3_fine_tune/weights/best.pt")

test_metrics = best_model.val(
    data=dataset_yaml,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    exist_ok=True,
    name="cfg3_test_eval"
)

print("\n================ CONFIG 3 TEST SET RESULTS ================")
print(f"Overall Test mAP50:    {test_metrics.box.map50:.4f}")
print(f"Overall Test mAP50-95: {test_metrics.box.map:.4f}\n")

# Class indices: 0=Cyclist, 1=Pedestrian, 2=Car
class_mapping = {0: 'Cyclist', 1: 'Pedestrian', 2: 'Car'}
for idx, c_name in class_mapping.items():
    print(f"Class: {c_name:<12} | AP50: {test_metrics.box.ap50[idx]:.4f} | AP50-95: {test_metrics.box.maps[idx]:.4f}")
print("===========================================================")


## 8. Training Curves & Test Set Plots (Config 3)

In [ ]:
print("=== Configuration 3: Training Convergence Curves ===")
# results.png shows the loss dropping and mAP rising over all 30 epochs
train_results_path = "runs/detect/cfg3_fine_tune/results.png"
if os.path.exists(train_results_path):
    display(Image(filename=train_results_path, width=800))
else:
    print("Could not find results.png")

print("\n=== Configuration 3: Test Set Precision-Recall Curve ===")
# BoxPR_curve.png from the final test evaluation
test_pr_path = "runs/detect/cfg3_test_eval/BoxPR_curve.png"
if os.path.exists(test_pr_path):
    display(Image(filename=test_pr_path, width=600))
else:
    print("Could not find BoxPR_curve.png")

print("\n=== Configuration 3: Test Set Confusion Matrix ===")
# Shows how well the model separated Cars, Pedestrians and Cyclists on the test set
test_cm_path = "runs/detect/cfg3_test_eval/confusion_matrix.png"
if os.path.exists(test_cm_path):
    display(Image(filename=test_cm_path, width=600))
else:
    print("Could not find confusion_matrix.png")


## 9. Save Best Weights to Google Drive

In [ ]:
import shutil

# 2. Copy the weight files to the Google Drive project folder
shutil.copy("runs/detect/cfg1_fine_tune/weights/best.pt", "/content/drive/MyDrive/Machine Learning Project/cfg1_best.pt")
shutil.copy("runs/detect/cfg3_fine_tune/weights/best.pt", "/content/drive/MyDrive/Machine Learning Project/cfg3_best.pt")

print("Saved cfg1_best.pt and cfg3_best.pt to Google Drive!")
